## Load Model and Evaluation


In [2]:
import joblib
import pandas as pd 
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, classification_report
from sklearn import metrics

In [3]:
# read the data
df_train = pd.read_csv("../data/df_train.csv")
df_test = pd.read_csv("../data/df_test.csv")

X_test = df_test['clean_text']
y_test = df_test['sentiment_label']

In [4]:
# read the saved model
def sparse_to_array(X):
    return X.toarray()

model_nb = joblib.load('../models/basic_model_nb.joblib')
model_lr = joblib.load('../models/basic_model_lr.joblib')
model_hg = joblib.load('../models/basic_model_hg.joblib')

In [ ]:
models = [model_nb, model_lr, model_hg]

In [6]:
for model in models:
    print(model.steps[1])

    print(confusion_matrix(y_test, model.predict(X_test)))
    print(classification_report(y_test, model.predict(X_test)))

('model', GaussianNB(var_smoothing=0.5))
[[  41    9  110]
 [  27   17  116]
 [  20   22 3957]]
              precision    recall  f1-score   support

    negative       0.47      0.26      0.33       160
     neutral       0.35      0.11      0.16       160
    positive       0.95      0.99      0.97      3999

    accuracy                           0.93      4319
   macro avg       0.59      0.45      0.49      4319
weighted avg       0.91      0.93      0.91      4319

('model', LogisticRegression())
[[  50   12   98]
 [  21   14  125]
 [  12    6 3981]]
              precision    recall  f1-score   support

    negative       0.60      0.31      0.41       160
     neutral       0.44      0.09      0.15       160
    positive       0.95      1.00      0.97      3999

    accuracy                           0.94      4319
   macro avg       0.66      0.47      0.51      4319
weighted avg       0.92      0.94      0.92      4319

('model', HistGradientBoostingClassifier())
[[  65   23

In [7]:
for model in models:
    print(model.steps[1])

    proba = model.predict_proba(X_test)
    
    # roc auc for multiclass: one vs rest -- good for imbalanced dataset
    roc_auc = metrics.roc_auc_score(y_test, proba, multi_class='ovr')
    print("ROC AUC: ", roc_auc)

    # since they have similar ovr --- should a tweak in threshold help? 


('model', GaussianNB(var_smoothing=0.5))
ROC AUC:  0.9330652065091091
('model', LogisticRegression())
ROC AUC:  0.9507577817903622
('model', HistGradientBoostingClassifier())
ROC AUC:  0.9324628275642084


In [40]:
# hmm ok ok?
proba = model_lr.predict_proba(X_test)
result = pd.DataFrame(proba)
# sns.kdeplot(result)

y_pred = result.idxmax(axis=1).map({0: 'negative', 1: 'neutral', 2: 'positive'})

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

# next, divide by prob/2 + 0.25: MID things. hehe. ok. right
divisor = df_train.sentiment_label.value_counts()/len(df_train)
# it's the nganu one hehe. 
divisor.sort_values(ascending=True, inplace=True)
divisor = divisor.values
# classify by the highest probability
y_pred = result.div(divisor).idxmax(axis=1).map({0: 'negative', 1: 'neutral', 2: 'positive'})

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

divisor = df_train.sentiment_label.value_counts()/len(df_train)/2 + 0.25
divisor.sort_values(ascending=True, inplace=True)
divisor = divisor.values
# classify by the highest probability
y_pred = result.div(divisor).idxmax(axis=1).map({0: 'negative', 1: 'neutral', 2: 'positive'})

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[  50   12   98]
 [  21   14  125]
 [  12    6 3981]]
              precision    recall  f1-score   support

    negative       0.60      0.31      0.41       160
     neutral       0.44      0.09      0.15       160
    positive       0.95      1.00      0.97      3999

    accuracy                           0.94      4319
   macro avg       0.66      0.47      0.51      4319
weighted avg       0.92      0.94      0.92      4319

[[ 114   44    2]
 [  57   91   12]
 [ 229  476 3294]]
              precision    recall  f1-score   support

    negative       0.28      0.71      0.41       160
     neutral       0.15      0.57      0.24       160
    positive       1.00      0.82      0.90      3999

    accuracy                           0.81      4319
   macro avg       0.48      0.70      0.51      4319
weighted avg       0.94      0.81      0.86      4319

[[  84   25   51]
 [  35   35   90]
 [  38   40 3921]]
              precision    recall  f1-score   support

    negative      

Kinda conclusion
- accuracy is pretty useless: all has similar accuracy (due to highly imbalanced dataset)
- rather we could see from macro averaged f1-score (all label are considered equal): lr and nb got similar performance, while hg is a bit better, especially dealing with minority classes (negative and neutral)
- but, considering it took 20 times longer training using hgb than lr and nb, then 
- let's choose lr instead for future use: in case need a retraining, well that's that

Improvement but probably not
- using threshold moving of such 
- using test data, not good, but at least that's that. 
- well lemme finish it later on lol. 

And bit of suggession
- remember how disgusting is it to use raw accuracy? 
- then, how about proposing a new metric to use: 
- either use weighted modeling, made up cost function, or even idk, can we use uhm, well 